In [1]:
import warnings
warnings.filterwarnings("ignore")
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from datasets import load_dataset, Audio
import torch
from IPython.display import Audio as display_Audio, display
import torchaudio

In [2]:
from transformers import BitsAndBytesConfig
import torch

In [3]:
device = torch.device('cpu')
torch_dtype = torch.float32

model_id = "openai/whisper-small"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True)

model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

In [4]:
pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    max_new_tokens=128,
    chunk_length_s=30,
    batch_size=16,
    return_timestamps=True,
    torch_dtype=torch_dtype,
    device=device,
)

In [5]:
#utility functions
def load_recorded_audio(path_audio,input_sample_rate=48000,output_sample_rate=16000):
    # Dataset: convert recorded audio to vector
    waveform, sample_rate = torchaudio.load(path_audio)
    waveform_resampled = torchaudio.functional.resample(waveform, orig_freq=input_sample_rate, new_freq=output_sample_rate) #change sample rate to 16000 to match training. 
    sample = waveform_resampled.numpy()[0]
    return sample

def run_inference(path_audio, output_lang, pipe):
    sample = load_recorded_audio(path_audio)
    result = pipe(sample, generate_kwargs = {"language": output_lang, "task": "transcribe"})
    print(result["text"])

In [6]:
path_audio = "data/SampleMockCallV2.wav"
output_lang = "en"
text = run_inference(path_audio,output_lang, pipe)

You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, None], [2, 50359]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


 Thank you for calling Lasada, my name is Chris, how can I help you today? I'm so disappointed with you all, I ordered a dozen wine glass and they came broken. Oh so sorry to really bet you received a broken item, we thank you for letting us know, let me check what are the available options for us. May I get your order number? 82993 Thank you to make sure I'm looking at the right account. Can you provide your full name and your phone number? Stella McCartney 5623738 Great, I can confirm that your package was successfully delivered last March 1 at 925 a.m. Are all glasses broken? Yes all of them as the bags damage as well it looked okay all right


In [7]:
inference_record = [
    {"LargeV3Turbo":{"AudioDuration":54,"InferenceTime":50}},
    {"LargeV3":{"AudioDuration":54,"InferenceTime":84}},
    {"Medium":{"AudioDuration":54,"InferenceTime":34}},
    {"Tiny":{"AudioDuration":54,"InferenceTime":2.2}},
    {"smaill":{"AudioDuration":54,"InferenceTime":10.8}},
                    ]

In [8]:
import pandas as pd

# Transforming the record into a DataFrame
data = {model_name: metrics for record in inference_record for model_name, metrics in record.items()}
df = pd.DataFrame.from_dict(data, orient='index')

# Resetting the index for better readability (optional)
df.index.name = 'Model'
df.reset_index(inplace=True)

print(df)

          Model  AudioDuration  InferenceTime
0  LargeV3Turbo             54           50.0
1       LargeV3             54           84.0
2        Medium             54           34.0
3          Tiny             54            2.2
4        smaill             54           10.8
